In [1]:
import os
os.chdir("../Datasets")

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier


In [3]:
#load data
bank = pd.read_csv('bank.csv')

In [4]:
# 2) Encode target and yes/no binaries
bank["deposit"] = bank["deposit"].map({"no": 0, "yes": 1})

yes_no_cols = ['default','housing','loan']
bank[yes_no_cols] = bank[yes_no_cols].replace({'yes':1,'no':0})

C:\Users\dbda.STUDENTSDC\AppData\Local\Temp\ipykernel_12484\3979867758.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bank[yes_no_cols] = bank[yes_no_cols].replace({'yes':1,'no':0})


In [5]:
#Separate features and target
X = bank.drop(columns='deposit')
y = bank['deposit']

In [6]:
# 4) Identify categorical and numeric columns
num_cols = X.select_dtypes(exclude=[object]).columns
cat_cols = X.select_dtypes(include=[object]).columns

In [7]:
# 5) Train/Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

In [8]:
# 6) One-hot encode categoricals -> pandas DataFrame
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False).set_output(transform='pandas')
X_train_enc = ohe.fit_transform(X_train[cat_cols])
X_test_enc = ohe.transform(X_test[cat_cols])

X_train_num = X_train[num_cols]
X_test_num = X_test[num_cols]

X_train_all = pd.concat([X_train_enc,X_train_num], axis=1)
X_test_all = pd.concat([X_test_enc,X_test_num], axis=1)

In [9]:
scaler = StandardScaler()
X_train_log = scaler.fit_transform(X_train_all)
X_test_log  = scaler.transform(X_test_all)

In [10]:
# ------------ Model 1: Logistic Regression ------------
log_clf = LogisticRegression(max_iter=1000)
log_clf.fit(X_train_log, y_train)
y_pred_log = log_clf.predict(X_test_log)

In [11]:
# ------------ Model 2: Decision Tree ------------
dt_clf = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_clf.fit(X_train_all, y_train)
y_pred_dt = dt_clf.predict(X_test_all)

In [12]:

print("Logistic first 10 preds :", y_pred_log[:10])
print("Tree first 10 preds     :", y_pred_dt[:10])

Logistic first 10 preds : [1 1 1 0 1 1 1 1 0 1]
Tree first 10 preds     : [1 1 1 0 1 1 0 1 0 1]


In [ ]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_test, y_pred_dt)
acc = accuracy_score(y_test, y_pred_log)